# Regressió ML amb Sklearn

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Càrrega de dades

Aquestes dades han d'estar netes i preparades per aplicar ML.

En aquest exemple de `tips`, volem predir la propina (`tip`) que pagarà un client.

In [4]:
df = sns.load_dataset('tips')
df.head()

,total_bill,tip,sex,smoker,day,time,size
0,16.99,1.01,Female,No,Sun,Dinner,2
1,10.34,1.66,Male,No,Sun,Dinner,3
2,21.01,3.50,Male,No,Sun,Dinner,3
3,23.68,3.31,Male,No,Sun,Dinner,2
4,24.59,3.61,Female,No,Sun,Dinner,4


In [5]:
df.shape

(244, 7)

## Pipeline de Feature Engineering

Defineix els passos de preprocessament per als models que ho necessiten.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [7]:
onehot_pipeline = Pipeline([
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore')),
])

In [8]:
onehot_features = ['sex', 'smoker', 'day', 'time']

preprocessor = ColumnTransformer(
    transformers=[
        ('onehot', onehot_pipeline, onehot_features),
    ],
    remainder='passthrough',
)

## Train / Test

- Podem emular dades no vistes dividint les nostres dades en conjunts de **train** i de **test**.
    - Usem el conjunt d'entrenament per ajustar (entrenar) el nostre model.
    - Usem el conjunt de test per avaluar el rendiment del model amb dades no vistes.
- Les mides de test habituals són del 10-30% de les dades inicials.
    - Els conjunts de dades grans poden requerir un percentatge menor (p. ex., 10-20%).
    - Els conjunts de dades petits poden necessitar un conjunt de test més gran per garantir que sigui representatiu (p. ex., 30-40%).

In [9]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import train_test_split, cross_validate, KFold

In [10]:
# Crear la variable X amb les característiques predictores (sense l'objectiu!)
X = df.drop(columns='tip')

# Separar l'objectiu en la variable `y`
y = df['tip']

# Divisió Entrenament / Test
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,  # 20% de les dades en el test
    shuffle=True,
    random_state=42
)

## Cross-validation

In [11]:
# Declare KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

## Entrenament del model

In [12]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

### ElasticNet

In [13]:
from sklearn.linear_model import ElasticNet
from sklearn.preprocessing import StandardScaler, RobustScaler, MinMaxScaler

In [17]:
# Definim una Pipeline que primer processarà les dades i llavors aplicarà LR
en_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('scaler', None),
    ('en', ElasticNet(max_iter=1000))
])

param_dist = {
    'scaler': [StandardScaler(), RobustScaler(), MinMaxScaler()],
    'en__l1_ratio': np.arange(0, 1.01, 0.01),
    'en__alpha': np.arange(0.01, 1.01, 0.01),
}

# Entrenar
en_rs = RandomizedSearchCV(
    en_pipeline,
    param_distributions=param_dist,
    n_iter=50,
    scoring='neg_mean_absolute_error',
    return_train_score=True,
    cv=kf,
    random_state=42,
    n_jobs=-1
)
en_rs.fit(X_train, y_train)

print('Best RandomizedSearchCV parameters: ', en_rs.best_params_)

Best RandomizedSearchCV parameters:  {'scaler': RobustScaler(), 'en__l1_ratio': np.float64(0.54), 'en__alpha': np.float64(0.03)}


In [20]:
print('CV Train MAE:', -en_rs.cv_results_['mean_train_score'][en_rs.best_index_].round(2))
print('CV Validation MAE:', -en_rs.cv_results_['mean_test_score'][en_rs.best_index_].round(2))

CV Train MAE: 0.76
CV Validation MAE: 0.8
